# Heart Disease Risk Predictor
Hi, My name is Joaquin Carmona, and I'll be working on this project to predict the risk of heart disease. This notebook will be used to explore the data and perform EDA. I'll be learning as I go and using this notebook to document my progress, videos or documents I find useful, and any other resources I find helpful.  

My purpose with this project is to jump in the Healthcare AI world doing this basic project to get started in this world. I'll be using different approaches and algorithms to predict the risk of heart disease.

## Dataset
The UCI Heart Disease dataset originates from the Cleveland Clinic Foundation (Detrano et al., 1989). 
It contains 14 clinical variables collected from 303 patients, with a binary target indicating 
presence or absence of heart disease.

## Reference
Detrano, R. et al. (1989). *International application of a new probability algorithm for the 
diagnosis of coronary artery disease.* American Journal of Cardiology, 64(5), 304-310.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('heart_disease_uci.csv')

# Dataframe info
print(df.shape)
print(df.dtypes)
df.head(10)

(920, 16)
id            int64
age           int64
sex             str
dataset         str
cp              str
trestbps    float64
chol        float64
fbs          object
restecg         str
thalch      float64
exang        object
oldpeak     float64
slope           str
ca          float64
thal            str
num           int64
dtype: object
id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0
5,6,56,Male,Cleveland,atypical angina,120.0,236.0,False,normal,178.0,False,0.8,upsloping,0.0,normal,0
6,7,62,Female,Cleveland,asymptomatic,140.0,268.0,False,lv hypertrophy,160.0,False,3.6,downsloping,2.0,normal,3
7,8,57,Female,Cleveland,asymptomatic,120.0,354.0,False,normal,163.0,True,0.6,upsloping,0.0,normal,0
8,9,63,Male,Cleveland,asymptomatic,130.0,254.0,False,lv hypertrophy,147.0,False,1.4,flat,1.0,reversable defect,2
9,10,53,Male,Cleveland,asymptomatic,140.0,203.0,True,lv hypertrophy,155.0,True,3.1,downsloping,0.0,reversable defect,1


In [ ]:
print(df.isnull().sum())
#first we check for null values, as we can see there are a lot of null values in the dataset, we will need to handle them, and take the best decision for each case.


id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64


In [5]:
print(df.columns)

Index(['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs',
       'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'],
      dtype='str')


## Dataset Variables 

The dataset contains 920 patients from 4 different institutions and 16 variables:

| Variable | Type | Clinical Description |
|----------|------|-------------------|
| `id` | int | Unique patient identifier. No predictive value. |
| `age` | int | Age in years. Well-established cardiovascular risk factor. |
| `sex` | str | Biological sex: Male / Female. Men face higher risk at younger ages. |
| `dataset` | str | Source institution: Cleveland, Hungary, Switzerland, VA Long Beach. |
| `cp` | str | Chest pain type: typical angina, atypical angina, non-anginal, asymptomatic. Asymptomatic pain is paradoxically the most associated with severe disease. |
| `trestbps` | float | Resting blood pressure at hospital admission (mm Hg). Hypertension is a major risk factor. |
| `chol` | float | Serum cholesterol (mg/dl). High levels associated with arterial plaque buildup. |
| `fbs` | object | Fasting blood sugar > 120 mg/dl: True/False. Indicates diabetes, a critical comorbidity. |
| `restecg` | str | Resting electrocardiogram results: normal, ST-T wave abnormality, left ventricular hypertrophy. |
| `thalch` | float | Maximum heart rate achieved during stress test. Lower capacity = higher risk. |
| `exang` | object | Exercise-induced angina: True/False. Pain triggered by exertion signals ischemia. |
| `oldpeak` | float | ST segment depression induced by exercise relative to rest. Indicates myocardial ischemia. |
| `slope` | str | Slope of the peak exercise ST segment: upsloping, flat, downsloping. Downsloping is the most concerning. |
| `ca` | float | Number of major vessels colored by fluoroscopy (0-3). More obstructed vessels = greater severity. |
| `thal` | str | Thallium stress test result (blood flow to heart): normal, fixed defect, reversible defect. |
| `num` | int | **Target variable.** Heart disease diagnosis: 0 = absent, 1-4 = present (severity). |
